# 🤖 Yapay Zeka ve Üretken Modeller Atölyesi — 4
**BTK Akademi | Dr. Öğr. Üyesi Furkan Göz**

---

## 📋 İçindekiler
1. [Görüntü Verisi ve Piksel Kavramı](#1)
2. [Grayscale ve RGB Görüntüler](#2)
3. [Klasik Sinir Ağları vs CNN](#3)
4. [Convolution (Evrişim) İşlemi](#4)
5. [Stride ve Padding](#5)
6. [Pooling (Havuzlama)](#6)
7. [Fashion MNIST ile CNN — Sınıflandırma](#7)
8. [CIFAR-10 ile CNN + Veri Artırma](#8)
9. [Dengesiz Veri Kümeleri](#9)
10. [YOLO ile Nesne Tespiti](#10)

---

<a id='1'></a>
## 1. 🖼️ Görüntü Verisi ve Piksel Kavramı

Bilgisayarlar görüntüleri **piksel matrisi** olarak temsil eder.

| Format | Boyut | Açıklama |
|--------|-------|----------|
| Grayscale | `(H, W)` | Her piksel 0–255 arası tek değer |
| RGB | `(H, W, 3)` | Her piksel R, G, B için 3 değer |

**Örnek piksel değerleri (RGB):**
- `(255, 0, 0)` → Saf Kırmızı  
- `(0, 255, 0)` → Saf Yeşil  
- `(128, 128, 128)` → Gri

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 28x28 boyutunda rastgele grayscale görüntü simülasyonu
grayscale_img = np.random.randint(0, 256, (28, 28), dtype=np.uint8)

print("Görüntü şekli (shape):", grayscale_img.shape)
print("Toplam piksel sayısı :", grayscale_img.size)
print("\nSol üst 5x5 piksel matrisi:")
print(grayscale_img[:5, :5])

<a id='2'></a>
## 2. 🎨 Grayscale ve RGB Görüntüler

- **2D (Grayscale):** Tek kanallı, `image[H][W]` — hücre başına 1 değer
- **3D (RGB):** Üç kanallı — `image[H][W][0]` Kırmızı, `[1]` Yeşil, `[2]` Mavi

In [ ]:
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist, cifar10

# --- MNIST (Grayscale) ---
(X_grey, _), _ = mnist.load_data()
grey_img = X_grey[0]
print("Grayscale şekli:", grey_img.shape)   # (28, 28)

# --- CIFAR-10 (RGB) ---
(X_rgb, _), _ = cifar10.load_data()
rgb_img = X_rgb[0]
print("RGB şekli        :", rgb_img.shape)   # (32, 32, 3)

# Görselleştirme
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(grey_img, cmap='gray')
axes[0].set_title("Grayscale (1 kanal)")
axes[0].axis('off')

axes[1].imshow(rgb_img)
axes[1].set_title("RGB (3 kanal)")
axes[1].axis('off')

plt.tight_layout()
plt.show()

<a id='3'></a>
## 3. ⚠️ Klasik Sinir Ağları (MLP) ve Görüntü İşleme Problemi

MLP görüntüleri işlemek için **flatten** (düzleştirme) yapar → **mekansal ilişkiler bozulur.**

```
(28×28) görüntü  →  784 boyutlu vektör
```

Bu nedenle **Convolutional Neural Networks (CNN)** geliştirilmiştir.  
CNN, komşuluk ilişkilerini koruyarak görüntüdeki desenleri öğrenir.

In [ ]:
import numpy as np

# MLP yaklaşımı: görüntü flatten ediliyor
sample_image = np.random.randint(0, 256, (28, 28))
flattened = sample_image.flatten()

print("Orijinal görüntü boyutu:", sample_image.shape)
print("Flatten sonrası        :", flattened.shape)
print("\nSorun: Piksel (3,4) ile (3,5) arasındaki mekansal komşuluk bilgisi kaybolur!")
print("Çözüm: CNN mimarisi mekansal yapıyı korur.")

<a id='4'></a>
## 4. 🔲 Convolution (Evrişim) İşlemi

- Görüntü üzerinde küçük bir **filtre (kernel)** matrisi kaydırılır
- Her konumda **noktasal çarpım** (element-wise) toplamı alınır → **özellik haritası (feature map)**
- CNN'de filtreler eğitim sırasında **otomatik öğrenilir** (klasik Sobel gibi el yapımı değil)

### Çıkış boyutu formülü (padding='valid'):
$$\text{Output} = \frac{\text{Input} - \text{Kernel} + 2P}{S} + 1$$

In [ ]:
import numpy as np

# Manuel 2D convolution (stride=1, padding='valid')
def manual_conv2d(image, kernel):
    ih, iw = image.shape
    kh, kw = kernel.shape
    oh = ih - kh + 1
    ow = iw - kw + 1
    output = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            output[i, j] = np.sum(image[i:i+kh, j:j+kw] * kernel)
    return output

# Slayttaki örnek giriş matrisi (7x7)
image = np.array([
    [5, 2, 6, 8, 2, 0, 1, 2],
    [4, 3, 4, 5, 1, 9, 6, 3],
    [3, 9, 2, 4, 7, 7, 6, 9],
    [1, 3, 4, 6, 8, 2, 2, 1],
    [8, 4, 6, 2, 3, 1, 8, 8],
    [5, 8, 9, 0, 1, 0, 2, 3],
    [9, 2, 6, 6, 3, 6, 2, 1],
    [9, 8, 8, 2, 6, 3, 4, 5]
])

# Slayttaki 3x3 kernel
kernel = np.array([
    [-1, 0, 1],
    [ 2, 1, 2],
    [ 1,-2, 0]
])

feature_map = manual_conv2d(image, kernel)

print("Giriş matrisi (8x8):")
print(image)
print("\nKernel (3x3):")
print(kernel)
print("\nÖzellik haritası (6x6):")
print(feature_map.astype(int))
print("\nİlk değer (slayttaki hesap):", int(feature_map[0, 0]))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import convolve
from tensorflow.keras.datasets import mnist

# MNIST'ten bir örnek al
(X, _), _ = mnist.load_data()
img = X[0].astype(np.float32)

# Klasik Sobel filtresi (dikey kenar tespiti)
sobel_vertical = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
])

filtered = convolve(img, sobel_vertical)

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(img, cmap='gray')
axes[0].set_title("Orijinal Görüntü")
axes[0].axis('off')

axes[1].imshow(np.abs(filtered), cmap='gray')
axes[1].set_title("Sobel Filtresi (Dikey Kenarlar)")
axes[1].axis('off')

plt.tight_layout()
plt.show()
print("Sobel filtresi:\n", sobel_vertical)
print("\nCNN'de bu tür filtreler eğitim sırasında veriden otomatik öğrenilir!")

<a id='5'></a>
## 5. 📐 Stride ve Padding

| Parametre | Açıklama |
|-----------|----------|
| `stride=1` | Filtre her adımda 1 hücre kayar |
| `stride=2` | Filtre 2 hücre atlar → çıkış küçülür |
| `padding='valid'` | Sıfır çerçeve eklenmez → çıkış küçülür |
| `padding='same'` | Sıfır çerçeve eklenir → boyut korunur |

In [ ]:
import tensorflow as tf
import numpy as np

# 5x5 giriş, 3x3 filtre — farklı stride/padding kombinasyonları
inp = np.random.rand(1, 5, 5, 1).astype(np.float32)  # (batch, H, W, channels)

configs = [
    {"padding": "valid", "strides": 1},
    {"padding": "same",  "strides": 1},
    {"padding": "same",  "strides": 2},
]

print(f"Giriş boyutu: {inp.shape[1]}x{inp.shape[2]}")
print(f"Filtre boyutu: 3x3\n")
print(f"{'Padding':<10} {'Stride':<8} {'Çıkış Boyutu'}")
print("-" * 35)

for c in configs:
    layer = tf.keras.layers.Conv2D(
        filters=1,
        kernel_size=(3, 3),
        strides=c["strides"],
        padding=c["padding"]
    )
    out = layer(inp)
    print(f"{c['padding']:<10} {c['strides']:<8} {out.shape[1]}x{out.shape[2]}")

<a id='6'></a>
## 6. 🏊 Pooling (Havuzlama) Katmanı

Pooling; özellik haritalarını **boyutunu küçültür**, hesaplama maliyetini düşürür.

| Tür | Açıklama |
|-----|----------|
| **Max Pooling** | Bölgedeki maksimum değeri alır — en çok tercih edilen |
| **Average Pooling** | Bölgedeki piksellerin ortalamasını alır |

**Boyut etkisi örneği:**  
`(64, 32, 32, 3)` → `MaxPooling2D(2,2)` → `(64, 16, 16, 3)`

In [ ]:
import numpy as np

# Slayttaki örnek 4x4 matris
feature_map = np.array([
    [29,  15,  28, 184],
    [ 0, 100,  70,  38],
    [12,  12,   7,   2],
    [12,  12,  45,   6]
])

def max_pooling_2x2(matrix):
    h, w = matrix.shape
    out = np.zeros((h // 2, w // 2), dtype=matrix.dtype)
    for i in range(0, h, 2):
        for j in range(0, w, 2):
            out[i//2, j//2] = np.max(matrix[i:i+2, j:j+2])
    return out

def avg_pooling_2x2(matrix):
    h, w = matrix.shape
    out = np.zeros((h // 2, w // 2), dtype=int)
    for i in range(0, h, 2):
        for j in range(0, w, 2):
            out[i//2, j//2] = int(np.mean(matrix[i:i+2, j:j+2]))
    return out

print("Orijinal feature map (4x4):")
print(feature_map)
print("\nMax Pooling (2x2) sonucu:")
print(max_pooling_2x2(feature_map))
print("\nAverage Pooling (2x2) sonucu:")
print(avg_pooling_2x2(feature_map))

<a id='7'></a>
## 7. 👗 Fashion MNIST ile CNN Sınıflandırma

**Fashion MNIST:** 10 giyim kategorisi, 28×28 grayscale, 70.000 görüntü

**Pipeline:**
1. Veri yükleme ve normalizasyon (`/ 255.0`)
2. Reshape: `(60000, 28, 28)` → `(60000, 28, 28, 1)` *(CNN 4D giriş bekler)*
3. One-Hot Encoding etiketler için
4. CNN mimarisi: Conv2D → MaxPooling → Conv2D → MaxPooling → Flatten → Dense → Softmax
5. Derleme: `adam` optimizer, `categorical_crossentropy` loss
6. Eğitim ve değerlendirme

In [ ]:
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import fashion_mnist

(X_train, y_train), _ = fashion_mnist.load_data()

class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

plt.figure(figsize=(12, 4))
for label in range(10):
    idx = list(y_train).index(label)
    plt.subplot(2, 5, label + 1)
    plt.imshow(X_train[idx], cmap="gray")
    plt.title(f"{label} - {class_names[label]}", fontsize=8)
    plt.axis("off")
plt.tight_layout()
plt.suptitle("Fashion MNIST — Veri Seti Sınıfları", y=1.02, fontsize=12)
plt.show()

In [ ]:
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical

(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

# 1. Normalizasyon: [0, 255] → [0, 1]
X_train = X_train / 255.0
X_test  = X_test  / 255.0

# 2. Reshape: (N, 28, 28) → (N, 28, 28, 1)  — CNN 4D giriş bekler
X_train = X_train.reshape(-1, 28, 28, 1)
X_test  = X_test.reshape(-1, 28, 28, 1)

# 3. One-Hot Encoding
y_train_cat = to_categorical(y_train, num_classes=10)
y_test_cat  = to_categorical(y_test,  num_classes=10)

print("X_train shape:", X_train.shape)
print("X_test  shape:", X_test.shape)
print("y_train (ilk 3, one-hot):\n", y_train_cat[:3])

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    Flatten(),
    Dense(128, activation='relu'),
    Dense(10,  activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    X_train, y_train_cat,
    epochs=10,
    batch_size=64,
    validation_data=(X_test, y_test_cat)
)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Doğruluk grafiği
axes[0].plot(history.history['accuracy'],     label='Eğitim Doğruluğu')
axes[0].plot(history.history['val_accuracy'], label='Doğrulama Doğruluğu')
axes[0].set_title('Doğruluk Grafiği')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Doğruluk')
axes[0].legend()

# Kayıp grafiği
axes[1].plot(history.history['loss'],     label='Eğitim Kaybı')
axes[1].plot(history.history['val_loss'], label='Doğrulama Kaybı')
axes[1].set_title('Kayıp Grafiği')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Kayıp')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

predictions      = model.predict(X_test)
predicted_classes = np.argmax(predictions, axis=1)

# Doğru tahminler
correct   = np.where(predicted_classes == y_test)[0]
incorrect = np.where(predicted_classes != y_test)[0]

print(f"Toplam test örneği : {len(y_test)}")
print(f"Doğru tahmin sayısı: {len(correct)}")
print(f"Yanlış tahmin sayısı: {len(incorrect)}")

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle("✅ Doğru Sınıflandırmalar", fontsize=12)
for i, ax in enumerate(axes.flatten()):
    idx = correct[i]
    ax.imshow(X_test[idx].reshape(28, 28), cmap='gray')
    ax.set_title(f"T:{class_names[predicted_classes[idx]]}\nG:{class_names[y_test[idx]]}",
                 fontsize=7)
    ax.axis('off')
plt.tight_layout()
plt.show()

fig2, axes2 = plt.subplots(2, 5, figsize=(12, 5))
fig2.suptitle("❌ Yanlış Sınıflandırmalar", fontsize=12)
for i, ax in enumerate(axes2.flatten()):
    idx = incorrect[i]
    ax.imshow(X_test[idx].reshape(28, 28), cmap='gray')
    ax.set_title(f"T:{class_names[predicted_classes[idx]]}\nG:{class_names[y_test[idx]]}",
                 fontsize=7, color='red')
    ax.axis('off')
plt.tight_layout()
plt.show()

<a id='8'></a>
## 8. 🚗 CIFAR-10 ile CNN + Veri Artırma (Data Augmentation)

**CIFAR-10:** 10 sınıf, 32×32 RGB, 60.000 görüntü (50K eğitim / 10K test)

**Sınıflar:** Airplane, Automobile, Bird, Cat, Deer, Dog, Frog, Horse, Ship, Truck

### Veri Artırma Teknikleri
| Teknik | Parametre | Amaç |
|--------|-----------|------|
| Döndürme | `rotation_range=15` | Eğik nesneleri tanıma |
| Yatay çevirme | `horizontal_flip=True` | Yön değişkenliği |
| Yakınlaştırma | `zoom_range=0.1` | Kısmi nesne tanıma |
| Kaydırma | `width/height_shift_range=0.1` | Konum bağımsızlığı |
| Parlaklık | `brightness_range=[0.8, 1.2]` | Işık koşulları |

In [ ]:
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt

(X_train, y_train), (X_test, y_test) = cifar10.load_data()

X_train = X_train / 255.0
X_test  = X_test  / 255.0

y_train_cat = to_categorical(y_train, 10)
y_test_cat  = to_categorical(y_test,  10)

class_names = ['Airplane','Automobile','Bird','Cat','Deer',
               'Dog','Frog','Horse','Ship','Truck']

# İlk 5 görseli önizle
plt.figure(figsize=(10, 2))
for i in range(5):
    plt.subplot(1, 5, i+1)
    plt.imshow(X_train[i])
    plt.title(class_names[y_train[i][0]], fontsize=9)
    plt.axis('off')
plt.suptitle("CIFAR-10 Örnek Görseller", y=1.05)
plt.tight_layout()
plt.show()

print("X_train shape:", X_train.shape)  # (50000, 32, 32, 3)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)
datagen.fit(X_train)

# Artırılmış görsel örnekleri
sample = X_train[0:1]  # ilk görsel
plt.figure(figsize=(12, 2))
plt.subplot(1, 6, 1)
plt.imshow(X_train[0])
plt.title("Orijinal", fontsize=8)
plt.axis('off')

for i, (batch, _) in enumerate(datagen.flow(sample, sample, batch_size=1)):
    if i == 5:
        break
    plt.subplot(1, 6, i+2)
    plt.imshow(batch[0])
    plt.title(f"Aug {i+1}", fontsize=8)
    plt.axis('off')

plt.suptitle("Veri Artırma Örnekleri", y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model_cifar = Sequential([
    Conv2D(32,  (3, 3), activation='relu', input_shape=(32, 32, 3)),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(64,  (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(128, (3, 3), activation='relu'),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(10,  activation='softmax')
])

model_cifar.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_cifar.summary()

In [ ]:
batch_size = 64
epochs     = 30

history_cifar = model_cifar.fit(
    datagen.flow(X_train, y_train_cat, batch_size=batch_size),
    steps_per_epoch=X_train.shape[0] // batch_size,
    epochs=epochs,
    validation_data=(X_test, y_test_cat)
)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_cifar.history['accuracy'],     label='Eğitim')
axes[0].plot(history_cifar.history['val_accuracy'], label='Doğrulama')
axes[0].set_title('CIFAR-10 Doğruluk')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Doğruluk')
axes[0].legend()

axes[1].plot(history_cifar.history['loss'],     label='Eğitim')
axes[1].plot(history_cifar.history['val_loss'], label='Doğrulama')
axes[1].set_title('CIFAR-10 Kayıp')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Kayıp')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

predictions   = model_cifar.predict(X_test)
pred_classes  = np.argmax(predictions, axis=1)
true_classes  = np.argmax(y_test_cat,  axis=1)

correct   = np.where(pred_classes == true_classes)[0]
incorrect = np.where(pred_classes != true_classes)[0]

# Doğru tahminler
fig, axes = plt.subplots(1, 5, figsize=(12, 3))
fig.suptitle("✅ Doğru Sınıflandırmalar (CIFAR-10)", fontsize=11)
for i, ax in enumerate(axes):
    idx = correct[i]
    ax.imshow(X_test[idx])
    ax.set_title(f"T:{class_names[pred_classes[idx]]}\nG:{class_names[true_classes[idx]]}",
                 fontsize=7)
    ax.axis('off')
plt.tight_layout(); plt.show()

# Yanlış tahminler
fig2, axes2 = plt.subplots(1, 5, figsize=(12, 3))
fig2.suptitle("❌ Yanlış Sınıflandırmalar (CIFAR-10)", fontsize=11)
for i, ax in enumerate(axes2):
    idx = incorrect[i]
    ax.imshow(X_test[idx])
    ax.set_title(f"T:{class_names[pred_classes[idx]]}\nG:{class_names[true_classes[idx]]}",
                 fontsize=7, color='red')
    ax.axis('off')
plt.tight_layout(); plt.show()

<a id='9'></a>
## 9. ⚖️ Dengesiz Veri Kümeleri

Gerçek dünyada sınıflar arası örnek sayısı eşit değildir (örn. tıbbi teşhis).  
Model, **çoğunluk sınıfına** matematiksel eğilim gösterir.

### Çözüm Yöntemleri
1. **Sınıf Ağırlıklandırma (Class Weighting):** Azınlık sınıfı hatası daha yüksek katsayıyla cezalandırılır
2. **Oversampling (Veri Artırma):** Azınlık sınıfına augmentation uygulanır
3. **Focal Loss:** Kolay örneklerin ağırlığı düşürülür, zor örneklere odaklanılır

In [ ]:
# Sınıf Ağırlıklandırma — Keras uygulaması
# Örnek senaryo: 0 = normal (900), 1 = hastalık (100)

import numpy as np

# Sahte dengesiz veri oluştur
np.random.seed(42)
n_normal   = 900
n_disease  = 100

X_imb = np.random.rand(n_normal + n_disease, 10)
y_imb = np.array([0] * n_normal + [1] * n_disease)

print("Sınıf dağılımı:")
print(f"  Sınıf 0 (Normal) : {np.sum(y_imb == 0)} örnek")
print(f"  Sınıf 1 (Hastalık): {np.sum(y_imb == 1)} örnek")

total = len(y_imb)
weight_0 = total / (2 * np.sum(y_imb == 0))
weight_1 = total / (2 * np.sum(y_imb == 1))

class_weights_dict = {0: weight_0, 1: weight_1}
print(f"\nHesaplanan sınıf ağırlıkları:")
print(f"  Sınıf 0 (Normal) : {weight_0:.2f}")
print(f"  Sınıf 1 (Hastalık): {weight_1:.2f}")
print("\nKullanım: model.fit(X, y, class_weight=class_weights_dict)")

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Basit bir Dense model (görüntü değil, kavramı göstermek için)
model_imb = Sequential([
    Dense(16, activation='relu', input_shape=(10,)),
    Dense(1,  activation='sigmoid')
])

model_imb.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# class_weight parametresi ile eğitim
history_imb = model_imb.fit(
    X_imb, y_imb,
    epochs=20,
    batch_size=32,
    class_weight=class_weights_dict,
    verbose=0
)

print("Eğitim tamamlandı!")
print(f"Son doğruluk: {history_imb.history['accuracy'][-1]:.4f}")

<a id='10'></a>
## 10. 🎯 YOLO ile Nesne Tespiti

**YOLO (You Only Look Once):** Tek bir ileri yayılım işlemiyle tüm nesneleri tespit eder.

- Görüntüyü **grid** yapısına böler
- Her hücre: sınıf tahmini + koordinatlar `(x, y, w, h)`
- Sınıflandırma + konumlandırma aynı anda
- **YOLO11 / YOLO26** (Ultralytics) — nano, small, medium, large, xlarge modelleri

### Görevler
| Görev | Açıklama |
|-------|----------|
| `detect` | Bounding box ile nesne tespiti |
| `classify` | Görüntü sınıflandırma |
| `segment` | Piksel bazlı maske |
| `track` | Video'da nesne takibi |
| `pose` | İskelet/eklem noktası tespiti |

### Metrikler
| Metrik | Açıklama |
|--------|----------|
| `mAP50` | IoU ≥ 0.50 için ortalama hassasiyet |
| `mAP50-95` | IoU 0.50–0.95 arası 11 noktanın ortalaması |
| `Precision` | Doğru tespit / toplam tahmin |
| `Recall` | Doğru tespit / toplam gerçek nesne |

In [ ]:
# YOLO kurulumu (Google Colab veya yerel ortam)
%pip install ultralytics -q

In [ ]:
import ultralytics
ultralytics.checks()

In [ ]:
# YOLO ile tahmin — CLI yöntemi
!yolo predict model=yolo11n.pt source='https://ultralytics.com/images/zidane.jpg' save=True

In [ ]:
from ultralytics import YOLO

# Model yükleme
model_yolo = YOLO('yolo11n.pt')  # önceden eğitilmiş ağırlıklar

# Tahmin
results = model_yolo('https://ultralytics.com/images/bus.jpg')

# Sonuçları incele
for r in results:
    print("Tespit edilen nesne sayısı:", len(r.boxes))
    if r.boxes is not None:
        for box in r.boxes:
            cls_id  = int(box.cls[0])
            conf    = float(box.conf[0])
            cls_name = model_yolo.names[cls_id]
            print(f"  Sınıf: {cls_name:<15} Güven: {conf:.2f}")

In [ ]:
# YOLO eğitimi — CLI yöntemi
# Not: Gerçek eğitim için GPU önerilir (Google Colab T4/A100)
!yolo train model=yolo11n.pt data=coco8.yaml epochs=3 imgsz=640

In [ ]:
# Python API ile eğitim, doğrulama ve dışa aktarma
from ultralytics import YOLO

model_yolo = YOLO('yolo11n.pt')

# Eğitim
results_train = model_yolo.train(
    data='coco8.yaml',
    epochs=3,
    imgsz=640
)

# Doğrulama
results_val = model_yolo.val()

# Modeli dışa aktarma (ONNX formatı)
# model_yolo.export(format='onnx')
print("\nDışa aktarma formatları: torchscript, onnx, openvino, coreml, tflite, ...")

In [ ]:
# YOLO etiket dosyası formatı açıklaması
# Her satır: class_id  x_center  y_center  width  height
# Koordinatlar görüntü boyutuna normalize edilmiştir [0, 1]

example_label = """
45  0.479492  0.688771  0.955609  0.5955
45  0.736516  0.247188  0.498875  0.476417
50  0.637063  0.732938  0.494125  0.510583
"""

print("Örnek YOLO etiket dosyası içeriği:")
print(example_label)
print("Format: class_id  x_center  y_center  width  height")
print("Not  : Tüm değerler görüntü genişlik/yüksekliğine normalize edilmiştir.")

---
## 📌 Özet

| Konu | Anahtar Bilgi |
|------|---------------|
| Görüntü Verisi | Piksel matrisi; Grayscale `(H,W)`, RGB `(H,W,3)` |
| CNN vs MLP | CNN mekansal ilişkileri korur, MLP flatten ile bozar |
| Convolution | Filtre × giriş → feature map; filtreler veriden öğrenilir |
| Stride/Padding | Stride: adım boyutu; Padding=same: boyutu korur |
| Pooling | MaxPooling boyutu küçültür, belirgin özellikleri saklar |
| Normalizasyon | `/255.0` → `[0,1]` aralığı; daha hızlı ve kararlı öğrenme |
| Data Augmentation | Döndürme, flip, zoom → overfitting azalır |
| Dengesiz Veri | class_weight, SMOTE, Focal Loss |
| YOLO | Tek geçişte nesne tespiti; mAP50/mAP50-95 metrikleri |

---
*BTK Akademi — Yapay Zeka ve Üretken Modeller Atölyesi 4*